# 6-3절 연습 문제 풀이

이 노트북은 6-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch06/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 6-3절 예제 - 오즈의 마법사 띄어쓰기 모델
from torch.utils.data import Dataset, DataLoader, random_split

OZ_PATH = '../../data/wonderful_wizard_of_oz.txt'

def load_text(path=OZ_PATH, strip_gutenberg=False):
    with open(path, encoding='utf-8-sig') as f:
        text = f.read()
    if strip_gutenberg:
        start = text.find('*** START')
        end = text.find('*** END')
        if start != -1: text = text[text.find('\n', start) + 1:]
        if end != -1: text = text[:text.find('*** END')]
    return text

def make_spacing_example(input_text):
    input_text = input_text.strip().lower()
    seq = ''.join(c for c in input_text if not c.isspace())
    marks = ''.join('1' if c.isspace() else '0' for c in input_text) + '1'
    import re as _re
    return seq, _re.sub('01+', '1', marks)

class SpacingDataset(Dataset):
    def __init__(self, text, vocab, window=32):
        self.x, self.y, self.vocab, self.window = [], [], vocab, window
        seq, tgt = make_spacing_example(text)
        for i in range(0, len(seq) - window, window):
            self.x.append([vocab.get(c, 0) for c in seq[i:i + window]])
            self.y.append([int(v) for v in tgt[i:i + window]])
    def __len__(self): return len(self.x)
    def __getitem__(self, i):
        return torch.tensor(self.x[i]), torch.tensor(self.y[i], dtype=torch.float32)

class SpacingLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden=64, cell='LSTM'):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        rnn_cls = nn.LSTM if cell == 'LSTM' else nn.GRU
        self.rnn = rnn_cls(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, 1)
    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.fc(out).squeeze(-1)

## 연습 6-9

과적합이 발생하면 학습을 조기 종료할 수 있도록 학습 과정을 수정해 SpacingLSTM을 학습해 보자. 이를 위해 앞에서 설명한 방법으로 훈련 데이터와 검증 데이터를 분리하고, 에포크마다 검증 손실을 계산하는 검증 과정도 함께 추가해야 한다. 참고로 이 문제와 다음 문제를 구현하는 데 필요한 코드가 깃허브 예제 노트북의 [그림 6-12]를 생성하는 셀에 포함되어 있지만, 코드를 확인하기 전 직접 구현해 보자.

In [ ]:
text = load_text()[:60000]
chars = sorted(set(''.join(c for c in text.lower() if not c.isspace())))
vocab = {c: i + 1 for i, c in enumerate(chars)}      # 0은 <pad>
dataset = SpacingDataset(text, vocab)
g = torch.Generator().manual_seed(SEED)
n_val = int(len(dataset) * 0.2)
train_set, valid_set = random_split(dataset, [len(dataset) - n_val, n_val], generator=g)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)

torch.manual_seed(SEED)
model = SpacingLSTM(len(vocab) + 1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

PATIENCE = 3
best, wait, best_state, best_epoch = float('inf'), 0, None, 0
for epoch in range(1, 31):
    model.train()
    for x, y in train_loader:
        loss = criterion(model(x.to(device)), y.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    model.eval(); tot = n = 0
    with torch.no_grad():
        for x, y in valid_loader:
            tot += criterion(model(x.to(device)), y.to(device)).item() * len(x); n += len(x)
    val = tot / n
    print(f'{epoch:2d} 검증 손실 {val:.4f}')
    if val < best:
        best, wait, best_epoch = val, 0, epoch
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'조기 종료 (최적 {best_epoch} 에포크)'); break
model.load_state_dict(best_state)

띄어쓰기는 각 위치가 '띈다/안 띈다'인 **이진 분류**이므로 `nn.BCEWithLogitsLoss`를 사용한다(시그모이드를 내부에 포함해 수치적으로 안정적이다). 나머지 조기 종료 절차는 4장과 같다.

## 연습 6-10

[도전 문제] [연습 문제 6-9]에서 조기 종료를 판단하는 데 사용하는 지표를 F1 점수로 바꾸고, 조기 종료 시점과 모델의 성능이 어떻게 바뀌는지 확인해 보자.

In [ ]:
def f1_score(model, loader):
    model.eval(); tp = fp = fn = 0
    with torch.no_grad():
        for x, y in loader:
            pred = (torch.sigmoid(model(x.to(device))) > 0.5).float().cpu()
            tp += ((pred == 1) & (y == 1)).sum().item()
            fp += ((pred == 1) & (y == 0)).sum().item()
            fn += ((pred == 0) & (y == 1)).sum().item()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

torch.manual_seed(SEED)
model_f1 = SpacingLSTM(len(vocab) + 1).to(device)
optimizer = torch.optim.Adam(model_f1.parameters(), lr=1e-3)
best_f1, wait, best_epoch = -1.0, 0, 0
for epoch in range(1, 31):
    model_f1.train()
    for x, y in train_loader:
        loss = criterion(model_f1(x.to(device)), y.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    f1 = f1_score(model_f1, valid_loader)
    print(f'{epoch:2d} 검증 F1 {f1:.4f}')
    if f1 > best_f1:
        best_f1, wait, best_epoch = f1, 0, epoch
    else:
        wait += 1
        if wait >= 3:
            print(f'조기 종료 (최적 {best_epoch} 에포크, F1 {best_f1:.4f})'); break

띄어쓰기 위치는 전체 글자 중 일부(약 15~20%)뿐이라 **클래스가 불균형**하다. 손실이나 정확도는 '전부 안 띈다'로 예측해도 높게 나오지만, F1 점수는 정밀도와 재현율을 함께 보므로 이런 편향을 잡아낸다.

그래서 F1 기준 조기 종료는 손실 기준보다 **늦게 멈추는 경향**이 있고, 실제 띄어쓰기 품질은 더 좋아지는 경우가 많다.

## 연습 6-11

[연습 문제 6-9]나 [연습 문제 6-10]에서 최적 에포크 모델로 띄어쓰기를 예측한 결과는 더 많은 에포크로 학습한 모델보다 성능이 나빠질 가능성이 크다. 그 이유가 무엇일지 설명해 보자.

### 풀이

조기 종료로 고른 모델의 **띄어쓰기 결과**가 더 오래 학습한 모델보다 나빠 보일 수 있는 이유는 두 가지다.

1. **검증 지표와 체감 품질의 불일치**
조기 종료는 검증 손실(또는 F1)이 가장 좋은 시점을 고른다. 그런데 이 지표는 전체 글자 위치의 평균이라, 사람이 눈으로 보는 '문장이 자연스러운가'와는 다르다. 손실이 조금 더 나빠지는 구간에서 오히려 자주 쓰는 단어의 띄어쓰기가 더 정확해질 수 있다.

2. **과적합이 이 과제에서는 유리하게 작동한다**
띄어쓰기는 같은 말뭉치 안에서 **같은 단어가 반복**된다. 더 오래 학습해 훈련 데이터를 외울수록 그 말뭉치의 띄어쓰기는 잘 맞힌다. 즉 과적합된 모델이 **같은 소설의 문장에 대해서는** 더 좋아 보인다.

정리하면 조기 종료가 지키려는 것은 **처음 보는 텍스트에 대한 일반화 성능**이다. 학습에 쓴 텍스트로 결과를 확인하면 과적합된 모델이 더 좋아 보이는 것이 자연스럽다.

## 연습 6-12

소설 <오즈의 마법사> 원문 텍스트 파일을 메모장 등의 프로그램으로 열어서 내용을 살펴보자. 텍스트 파일의 첫 부분과 끝부분이 소설과 무엇이 다른지 확인한 후, 그 차이가 모델의 성능에 어떤 영향을 미칠지 예상해 보자. 그다음 파일의 첫 부분과 끝부분을 제거한 텍스트로 모델을 학습하고 결과를 확인해 보자.

In [ ]:
raw = load_text()
print('=== 파일 첫 부분 ===')
print(raw[:300])
print('\n=== 파일 끝부분 ===')
print(raw[-300:])

In [ ]:
# 프로젝트 구텐베르크 머리말·꼬리말을 제거하고 다시 학습한다.
clean = load_text(strip_gutenberg=True)[:60000]
print(f'원본 {len(raw):,}자 -> 정리 후 {len(load_text(strip_gutenberg=True)):,}자')

dataset_c = SpacingDataset(clean, vocab)
n_val = int(len(dataset_c) * 0.2)
tr_c, va_c = random_split(dataset_c, [len(dataset_c) - n_val, n_val],
                          generator=torch.Generator().manual_seed(SEED))
loader_c = DataLoader(tr_c, batch_size=32, shuffle=True)
valid_c = DataLoader(va_c, batch_size=32)

torch.manual_seed(SEED)
model_c = SpacingLSTM(len(vocab) + 1).to(device)
optimizer = torch.optim.Adam(model_c.parameters(), lr=1e-3)
for epoch in range(1, 11):
    model_c.train()
    for x, y in loader_c:
        loss = criterion(model_c(x.to(device)), y.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
print(f'정리한 텍스트로 학습한 모델의 검증 F1: {f1_score(model_c, valid_c):.4f}')

파일 앞뒤에는 **프로젝트 구텐베르크의 라이선스 안내문**이 붙어 있다. 소설 본문과 문체·어휘가 전혀 다른 법률 문구라, 이 부분까지 학습하면 모델이 소설과 무관한 패턴을 함께 배워 성능이 떨어진다.

실무에서 데이터 전처리의 첫 단계가 **불필요한 머리말·꼬리말 제거**인 이유다.

## 연습 6-13

<pad> 토큰을 사용해 입력 문자열 앞부분의 띄어쓰기도 예측할 수 있도록 데이터셋 클래스를 수정한 후 SpacingLSTM을 학습하고 결과를 확인해 보자.

In [ ]:
# 입력 앞에 <pad>를 채워 문장 첫 부분의 띄어쓰기도 예측하게 한다.
class PaddedSpacingDataset(Dataset):
    def __init__(self, text, vocab, window=32, pad_size=4):
        self.x, self.y = [], []
        seq, tgt = make_spacing_example(text)
        seq = '\0' * pad_size + seq          # 앞쪽에 <pad> 자리 확보
        tgt = '0' * pad_size + tgt
        for i in range(0, len(seq) - window, window):
            self.x.append([0 if c == '\0' else vocab.get(c, 0) for c in seq[i:i + window]])
            self.y.append([int(v) for v in tgt[i:i + window]])
    def __len__(self): return len(self.x)
    def __getitem__(self, i):
        return torch.tensor(self.x[i]), torch.tensor(self.y[i], dtype=torch.float32)

ds = PaddedSpacingDataset(load_text(strip_gutenberg=True)[:60000], vocab)
loader = DataLoader(ds, batch_size=32, shuffle=True)
torch.manual_seed(SEED)
model_p = SpacingLSTM(len(vocab) + 1).to(device)
optimizer = torch.optim.Adam(model_p.parameters(), lr=1e-3)
for epoch in range(5):
    model_p.train()
    for x, y in loader:
        loss = criterion(model_p(x.to(device)), y.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
print(f'학습 완료 - 최종 손실 {loss.item():.4f}')

`<pad>`에 어휘 사전 인덱스 0을 배정하고 입력 앞을 채우면, 윈도우 첫 글자에도 왼쪽 문맥이 생겨 문장 앞부분의 띄어쓰기를 예측할 수 있다. 임베딩 계층에 `padding_idx=0`을 지정하면 `<pad>`가 학습에 영향을 주지 않는데, 이 인자는 10장에서 자세히 다룬다.

## 연습 6-14

LSTM과 함께 소개한 GRU 계층을 사용해 띄어쓰기 모델을 정의하고 학습해 보자. LSTM을 사용한 모델과 어떤 차이가 있는지를 학습 시간까지 포함해 검토해 보자.

In [ ]:
import time
for cell in ('LSTM', 'GRU'):
    torch.manual_seed(SEED)
    model_c = SpacingLSTM(len(vocab) + 1, cell=cell).to(device)
    optimizer = torch.optim.Adam(model_c.parameters(), lr=1e-3)
    n_param = sum(p.numel() for p in model_c.parameters())
    start = time.time()
    for epoch in range(5):
        model_c.train()
        for x, y in train_loader:
            loss = criterion(model_c(x.to(device)), y.to(device))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    elapsed = time.time() - start
    print(f'{cell}: 파라미터 {n_param:,}개, 5 에포크 {elapsed:.1f}초, '
          f'검증 F1 {f1_score(model_c, valid_loader):.4f}')

GRU는 LSTM의 세 관문(입력·망각·출력)을 두 개(리셋·업데이트)로 줄이고 셀 상태를 없앤 구조다. 그래서 **파라미터가 약 25% 적고 학습이 빠르다**. 성능은 과제에 따라 다르지만, 이 정도 규모의 띄어쓰기 문제에서는 거의 차이가 없다. 자원이 제한적이라면 GRU가 합리적인 선택이다.

## 연습 6-15

[도전 문제] 소설 <오즈의 마법사>를 학습해 <오즈의 마법사> 스타일의 소설을 쓰는 모델을 만들어 보자. 6-2절과 6-3절의 예제 코드를 참고해 만들면 되는데, 어휘 사전을 구성하는 토큰의 단위(글자 또는 어절 등)나 하이퍼파라미터 등의 조건을 바꿔 보며 다양한 모델을 만들어 보거나 마중물 텍스트를 바꿔 보면서 결과가 어떻게 달라지는지 확인해 보자.

6장 학습 노트

In [ ]:
# 오즈의 마법사 스타일 문장 생성기 (글자 단위)
text = load_text(strip_gutenberg=True)[:100000].lower()
chars = sorted(set(text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
WINDOW = 40
idx = torch.tensor([c2i[c] for c in text])
xs = torch.stack([idx[i:i + WINDOW] for i in range(0, len(idx) - WINDOW - 1, 3)])
ys = torch.stack([idx[i + 1:i + WINDOW + 1] for i in range(0, len(idx) - WINDOW - 1, 3)])
loader = DataLoader(list(zip(xs, ys)), batch_size=64, shuffle=True)
print(f'어휘 {len(chars)}자, 샘플 {len(xs):,}개')

class OzWriter(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)
    def forward(self, x, state=None):
        out, state = self.rnn(self.emb(x), state)
        return self.fc(out), state

torch.manual_seed(SEED)
writer = OzWriter(len(chars)).to(device)
criterion_ce = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(writer.parameters(), lr=2e-3)
for epoch in range(1, 11):
    writer.train(); tot = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out, _ = writer(x)
        loss = criterion_ce(out.reshape(-1, len(chars)), y.reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot += loss.item(); n += 1
    print(f'{epoch:2d}/10 손실 {tot / n:.4f}')

In [ ]:
@torch.no_grad()
def write(seed_text, length=300, temperature=0.8):
    writer.eval()
    state, out = None, list(seed_text)
    x = torch.tensor([[c2i.get(c, 0) for c in seed_text]], device=device)
    for _ in range(length):
        logits, state = writer(x, state)
        probs = torch.softmax(logits[0, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1).item()
        out.append(i2c[nxt])
        x = torch.tensor([[nxt]], device=device)
    return ''.join(out)

print(write('dorothy '))

글자 단위 생성 모델의 요령은 세 가지다.

1. **입력과 정답을 한 칸씩 밀어** (x[i] → y[i]=x[i+1]) 매 시점마다 다음 글자를 맞히게 한다.
2. **온도(temperature)** 로 무작위성을 조절한다. 낮으면 반복적이고 안전한 문장, 높으면 다양하지만 엉뚱한 문장이 나온다.
3. 생성 시에는 **숨겨진 상태를 이어받아** 한 글자씩 넣는다.

어절 단위로 바꾸면 문법은 좋아지지만 어휘 사전이 커지고 처음 보는 단어를 만들 수 없다.